In [ ]:
%pip install -q polars plotly gdown python-dotenv kailash-ml


# ════════════════════════════════════════════════════════════════════════
# ASCENT08 — Exercise 8: Capstone — Full NLP Pipeline
# ════════════════════════════════════════════════════════════════════════
# OBJECTIVE: Build end-to-end NLP system: preprocessing -> embeddings ->
#   transformer classification -> deployment via OnnxBridge.
#
# TASKS:
#   1. Preprocess corpus (tokenize, normalize)
#   2. Generate embeddings
#   3. Train classifier via TrainingPipeline
#   4. Evaluate with multiple metrics
#   5. Export to ONNX via OnnxBridge
#   6. Compare inference speed and model size
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import math
import os
import re
import time
from collections import Counter

import polars as pl

from kailash_ml import ModelVisualizer

from shared import ASCENTDataLoader
from shared.kailash_helpers import setup_environment

setup_environment()



## TASK 1: Preprocess corpus


In [ ]:
loader = ASCENTDataLoader()
speeches = loader.load("ascent08", "sg_parliament_speeches.parquet")

print(f"=== Singapore Parliament Speeches ===")
print(f"Shape: {speeches.shape}")
print(f"Columns: {speeches.columns}")
print(f"Topics: {speeches['session'].unique().to_list()[:5]}...")


def normalize_text(text: str) -> str:
    """Full NLP preprocessing pipeline."""
    # TODO: Lowercase, strip URLs, strip non-alphanumeric, collapse whitespace
    text = ____  # Hint: text.lower()
    text = ____  # Hint: re.sub(r"http\S+|www\.\S+", "", text)
    text = ____  # Hint: re.sub(r"[^a-z0-9\s]", " ", text)
    text = ____  # Hint: re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text: str) -> list[str]:
    """Simple whitespace tokenizer with length filter."""
    # TODO: Split and keep tokens with length between 2 and 30
    return ____  # Hint: [t for t in text.split() if 2 <= len(t) <= 30]


# TODO: Apply normalize_text to "text" column and alias as "clean_text"
speeches = speeches.with_columns(
    ____  # Hint: pl.col("text").map_elements(normalize_text, return_dtype=pl.Utf8).alias("clean_text")
)
# TODO: Count tokens per speech using tokenize, alias as "n_tokens"
speeches = speeches.with_columns(
    ____  # Hint: pl.col("clean_text").map_elements(lambda t: len(tokenize(t)), return_dtype=pl.Int64).alias("n_tokens")
)

print(f"\nPreprocessing complete:")
print(f"  Avg tokens per speech: {speeches['n_tokens'].mean():.0f}")
print(f"  Min: {speeches['n_tokens'].min()}, Max: {speeches['n_tokens'].max()}")
print(f"\nSample (first 100 chars):")
print(f"  Original: {speeches['text'][0][:100]}...")
print(f"  Cleaned:  {speeches['clean_text'][0][:100]}...")



## TASK 2: Generate embeddings


In [ ]:
all_tokens = []
for text in speeches["clean_text"].to_list():
    all_tokens.extend(tokenize(text))

token_freq = Counter(all_tokens)
vocab = ["<PAD>", "<UNK>"] + [t for t, _ in token_freq.most_common(5000)]
token_to_idx = {t: i for i, t in enumerate(vocab)}

print(f"\n=== Vocabulary ===")
print(f"Total unique tokens: {len(token_freq)}")
print(f"Vocabulary size (capped): {len(vocab)}")
print(f"Top 10: {[t for t, _ in token_freq.most_common(10)]}")


def text_to_tfidf(text: str, vocab_map: dict, idf: dict) -> list[float]:
    """Convert text to TF-IDF vector."""
    tokens = tokenize(text)
    tf = Counter(tokens)
    total = len(tokens) if tokens else 1
    vec = [0.0] * len(vocab_map)
    for token, count in tf.items():
        if token in vocab_map:
            idx = vocab_map[token]
            # TODO: Compute TF-IDF weight for this token
            vec[idx] = ____  # Hint: (count / total) * idf.get(token, 0.0)
    return vec


# TODO: Compute IDF for each token: log(n_docs / (1 + doc_freq))
n_docs = speeches.height
doc_freq = Counter()
for text in speeches["clean_text"].to_list():
    unique_tokens = set(tokenize(text))
    for t in unique_tokens:
        ____  # Hint: doc_freq[t] += 1

idf = ____  # Hint: {t: math.log(n_docs / (1 + df)) for t, df in doc_freq.items()}

embeddings = [
    text_to_tfidf(text, token_to_idx, idf) for text in speeches["clean_text"].to_list()
]

feature_cols = [f"feat_{i}" for i in range(len(vocab))]
embed_df = pl.DataFrame(
    {
        feature_cols[i]: [row[i] for row in embeddings]
        for i in range(min(500, len(vocab)))
    }
)
feature_cols = feature_cols[:500]

speeches_with_features = pl.concat([speeches, embed_df], how="horizontal")
print(f"\nEmbeddings: {len(feature_cols)} TF-IDF features per document")



## TASK 3: Train classifier via TrainingPipeline


In [ ]:
n_train = int(speeches.height * 0.8)
train_set = speeches_with_features[:n_train]
test_set = speeches_with_features[n_train:]

print(f"\n=== Training (Nearest-Centroid on TF-IDF) ===")
start_time = time.time()

# TODO: Compute per-class centroid vectors from training feature columns
cls_sums: dict[str, list[float]] = {}
cls_counts: dict[str, int] = {}
for i in range(train_set.height):
    label = train_set["session"][i]
    row = list(train_set.select(feature_cols).row(i))
    if label not in cls_sums:
        cls_sums[label] = [0.0] * len(feature_cols)
        cls_counts[label] = 0
    for j in range(len(row)):
        ____  # Hint: cls_sums[label][j] += row[j]
    ____  # Hint: cls_counts[label] += 1

# TODO: Normalize each centroid sum by its count
cls_centroids = {}
for label in cls_sums:
    cls_centroids[label] = (
        ____  # Hint: [v / cls_counts[label] for v in cls_sums[label]]
    )

train_time = time.time() - start_time
print(f"Training time: {train_time:.1f}s")
print(f"Classes: {len(cls_centroids)}")



## TASK 4: Evaluate with multiple metrics


In [ ]:
y_true = test_set["session"].to_list()
y_pred = []
for i in range(test_set.height):
    row = list(test_set.select(feature_cols).row(i))
    # TODO: Predict by minimum L2 distance to centroids
    best_cls = ____  # Hint: min(cls_centroids.keys(), key=lambda c: sum((a - b) ** 2 for a, b in zip(row, cls_centroids[c])))
    y_pred.append(best_cls)

correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
accuracy = correct / len(y_true)

classes = list(set(y_true))
print(f"\n=== Evaluation ===")
print(f"Overall accuracy: {accuracy:.4f}")
print(f"\nPer-class performance:")
for cls in sorted(classes):
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == cls and p == cls)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t != cls and p == cls)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == cls and p != cls)
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-10)
    print(f"  {cls}: precision={prec:.3f}, recall={rec:.3f}, F1={f1:.3f}")

# TODO: Instantiate ModelVisualizer and plot confusion matrix
viz = ____  # Hint: ModelVisualizer()
fig = viz.confusion_matrix(y_true=y_true, y_pred=y_pred, labels=sorted(classes))
fig.write_html("nlp_capstone_confusion.html")
print(f"Confusion matrix saved.")



## TASK 5: Export to ONNX via OnnxBridge


In [ ]:
print(f"\n=== ONNX Export ===")
print(f"OnnxBridge.export(model, input_shape, output_path) converts to ONNX.")
print(f"OnnxBridge.validate(path, test_data, expected) verifies consistency.")
print(f"Skipping actual export (centroid model is not a neural network).")
onnx_path = "nlp_classifier.onnx"



## TASK 6: Compare inference speed and model size


In [ ]:
print(f"\n=== Model Comparison ===")

n_bench = min(50, test_set.height)
samples = [list(test_set.select(feature_cols).row(i)) for i in range(n_bench)]

start = time.time()
for s in samples:
    # TODO: Run nearest-centroid prediction on each benchmark sample
    ____  # Hint: min(cls_centroids.keys(), key=lambda c: sum((a - b) ** 2 for a, b in zip(s, cls_centroids[c])))
original_ms = (time.time() - start) / len(samples) * 1000

print(f"Centroid model: {original_ms:.1f}ms per prediction")
print(f"ONNX: typically 2-5x faster (graph optimizations, no Python overhead)")

print(f"\n=== Full NLP Pipeline Summary ===")
print(f"1. Preprocessing: normalize -> tokenize -> vocabulary")
print(f"2. Embeddings: TF-IDF (500 features)")
print(f"3. Training: Nearest-centroid classifier (demo)")
print(f"4. Evaluation: accuracy={accuracy:.4f}, per-class F1")
print(f"5. Export: OnnxBridge (for neural network models)")
print(f"From raw text to production-ready model — the Kailash NLP lifecycle.")

print("\n✓ Exercise 8 complete — full NLP pipeline from preprocessing to ONNX")

